# 02 — Leakage-safe baseline models

This notebook presents artifacts produced by the packaged training command. It does not reimplement model fitting.

Run this first if artifacts are absent:

```powershell
$env:PYTHONPATH = "src"
python -m sensorbudget.modeling.train
```

In [ ]:
# Import tabular, metric, and interactive plotting tools.
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import precision_recall_curve

PLOTLY_TEMPLATE = "plotly_white"
FEATURE_COLORS = {"all_sensors": "#4C78A8", "no_light": "#F58518"}

In [ ]:
# Locate the repository independently of the Jupyter launch directory.
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate pyproject.toml")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "models" / "baseline"

required = [
    "cv_fold_metrics.csv",
    "cv_summary.csv",
    "heldout_metrics.csv",
    "heldout_predictions.csv",
]
missing = [name for name in required if not (ARTIFACT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Run python -m sensorbudget.modeling.train; missing: " + ", ".join(missing)
    )

In [ ]:
# Load generated results; model artifacts remain outside version control.
cv_folds = pd.read_csv(
    ARTIFACT_DIR / "cv_fold_metrics.csv",
    parse_dates=["validation_start", "validation_end"],
)
cv_summary = pd.read_csv(ARTIFACT_DIR / "cv_summary.csv")
heldout = pd.read_csv(ARTIFACT_DIR / "heldout_metrics.csv")
predictions = pd.read_csv(
    ARTIFACT_DIR / "heldout_predictions.csv",
    parse_dates=["date"],
)

display(
    cv_summary.sort_values(["feature_set", "f1_mean"], ascending=[True, False])[
        ["feature_set", "model", "f1_mean", "f1_std", "precision_mean", "recall_mean"]
    ]
)

## Chronological cross-validation

Error bars show one standard deviation across five expanding validation folds. One fold is an all-unoccupied weekend, so every model has F1 equal to zero there.

In [ ]:
# Compare candidate mean F1 within each feature configuration.
fig = make_subplots(rows=1, cols=2, subplot_titles=("All sensors", "No Light"))
for column, feature_set in enumerate(["all_sensors", "no_light"], start=1):
    subset = cv_summary.loc[cv_summary["feature_set"] == feature_set].sort_values("f1_mean")
    fig.add_trace(
        go.Bar(
            x=subset["f1_mean"],
            y=subset["model"],
            orientation="h",
            error_x={"type": "data", "array": subset["f1_std"]},
            marker_color=FEATURE_COLORS[feature_set],
            text=subset["f1_mean"].map(lambda value: f"{value:.3f}"),
            textposition="inside",
            showlegend=False,
        ),
        row=1,
        col=column,
    )
fig.update_xaxes(title_text="Mean validation F1", range=[0, 1.25])
fig.update_layout(template=PLOTLY_TEMPLATE, title="Baseline model comparison", height=500)
fig.show()

In [ ]:
# Expose temporal variability rather than relying only on the mean.
selected_pairs = {
    ("all_sensors", "hist_gradient_boosting"),
    ("no_light", "logistic_regression"),
}
fig = go.Figure()
for feature_set, model in selected_pairs:
    subset = cv_folds.loc[
        (cv_folds["feature_set"] == feature_set) & (cv_folds["model"] == model)
    ]
    fig.add_trace(
        go.Scatter(
            x=subset["fold"],
            y=subset["f1"],
            mode="lines+markers",
            name=feature_set,
            marker_color=FEATURE_COLORS[feature_set],
            customdata=subset[["validation_occupied_rate"]],
            hovertemplate="Fold %{x}<br>F1: %{y:.3f}<br>Occupied: %{customdata[0]:.1%}<extra>%{fullData.name}</extra>",
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Selected models across chronological folds",
    xaxis_title="Validation fold",
    yaxis_title="F1",
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(range=[0, 1.05])
fig.show()

## Held-out evaluation

Only the selected model for each feature configuration is evaluated on the two supplied test periods.

In [ ]:
# Compare threshold metrics across test periods and feature configurations.
metric_names = ["precision", "recall", "f1", "balanced_accuracy"]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Test 1", "Test 2"))
for column, split in enumerate(["test_1", "test_2"], start=1):
    subset = heldout.loc[heldout["split"] == split]
    for _, row in subset.iterrows():
        fig.add_trace(
            go.Bar(
                x=metric_names,
                y=[row[metric] for metric in metric_names],
                name=row["feature_set"],
                legendgroup=row["feature_set"],
                showlegend=column == 1,
                marker_color=FEATURE_COLORS[row["feature_set"]],
            ),
            row=1,
            col=column,
        )
fig.update_yaxes(title_text="Score", range=[0, 1.05])
fig.update_layout(template=PLOTLY_TEMPLATE, title="Held-out threshold metrics", barmode="group")
fig.show()

display(
    heldout[
        ["feature_set", "model", "split", "precision", "recall", "f1", "average_precision", "roc_auc", "brier_score"]
    ]
)

In [ ]:
# Plot precision-recall curves from saved held-out probabilities.
fig = make_subplots(rows=1, cols=2, subplot_titles=("Test 1", "Test 2"))
for column, split in enumerate(["test_1", "test_2"], start=1):
    for feature_set in ["all_sensors", "no_light"]:
        subset = predictions.loc[
            (predictions["source_split"] == split)
            & (predictions["feature_set"] == feature_set)
        ]
        precision, recall, _ = precision_recall_curve(
            subset["Occupancy"], subset["probability_occupied"]
        )
        fig.add_trace(
            go.Scatter(
                x=recall,
                y=precision,
                mode="lines",
                name=feature_set,
                legendgroup=feature_set,
                showlegend=column == 1,
                line_color=FEATURE_COLORS[feature_set],
            ),
            row=1,
            col=column,
        )
fig.update_xaxes(title_text="Recall", range=[0, 1])
fig.update_yaxes(title_text="Precision", range=[0, 1.05])
fig.update_layout(template=PLOTLY_TEMPLATE, title="Held-out precision-recall curves")
fig.show()

## Conclusions

- Histogram gradient boosting is the strongest all-sensor baseline.
- Logistic regression is the strongest no-Light baseline by chronological mean F1.
- The all-sensor model remains strong across both tests.
- The no-Light model deteriorates sharply in `test_2`, especially in precision.
- Threshold 0.5 is only a baseline; tuning must use training-period validation, not these held-out results.
- The next experiment should compare additional sensor subsets and quantify the performance–cost frontier.